# Covertype Classification using Logistic Regression and LIME

## Objective

The objective of this activity is to train a Logistic Regression model using the Covertype dataset to predict forest cover type from cartographic and environmental features. 

The general performance of the model will be evaluated using accuracy, precision, recall, F1-score and a confusion matrix. Explainable AI techniques will then be used to investigate how the model makes its predictions. LIME will be used for a local explanation, while Logistic Regression coefficients will be examined as a global explanation.

# 1. Setup

In [ ]:
# If needed: %pip install scikit-learn pandas matplotlib lime scipy


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from IPython.display import display


# 2. Load and Explore the Dataset

In [ ]:
import sys
from pathlib import Path

for _directory in (Path.cwd(), Path.cwd() / "experiments", Path.cwd() / "XAI" / "experiments"):
    if (_directory / "controlled_lime_protocol.py").is_file():
        sys.path.insert(0, str(_directory))
        break
else:
    raise FileNotFoundError("Keep controlled_lime_protocol.py beside the experiment notebooks.")

from controlled_lime_protocol import (
    CLASS_NAMES, RANDOM_STATE, SAMPLE_SIZE, TEST_SIZE, BACKGROUND_SIZE,
    SAMPLE_BUDGETS, REPEATS, EXPLANATIONS_PER_CLASS, NUM_FEATURES, TOP_K,
    FEATURE_COUNTS, load_controlled_covtype, selected_explanation_cases,
    benchmark_lime, evaluate_predictive_performance,
)

X, y, X_train, X_test, y_train, y_test = load_controlled_covtype()
feature_names = X.columns.tolist()
class_names = list(CLASS_NAMES)
lime_background = X_train.sample(n=min(BACKGROUND_SIZE, len(X_train)), random_state=RANDOM_STATE)
explanation_cases = selected_explanation_cases(X_test, y_test)

print(f"Shared modeling sample: {len(X):,} rows, {X.shape[1]} features")
print(f"Train/test: {len(X_train):,} / {len(X_test):,} rows; seed: {RANDOM_STATE}")
print(f"LIME background: {len(lime_background):,} rows")
print(f"Cases: {len(explanation_cases)} ({EXPLANATIONS_PER_CLASS} per true class)")
print("Case indices:", explanation_cases.index.tolist())
print("Perturbation dimensions:", FEATURE_COUNTS)
print("Budgets:", SAMPLE_BUDGETS, "Repeats:", REPEATS, "Explanation features:", NUM_FEATURES)
X.head()

print("Target distribution:", y.value_counts().sort_index())


In [ ]:
print(X.columns.tolist())

## Dataset Description 

The Forest Covertype dataset contains cartographic information collected from forested areas in the Roosevelt National Forest in Colorado. 

The objective is to predict the dominant forest cover type of an area using geographical and environmental characteristics. 

The dataset contains 581,012 observations and 54 input features. 

The first 10 features are numerical variables describing characteristics such as:
- Elevation
- Aspect
- Slope
- Distance to hydrology
- Distance to roads
- Hillshade
- Distance to fire ignition points

The remaining 44 features are binary variables representing four wilderness areas and 40 soil types.

The target variable contains seven possible forest cover types, making this a multiclass classification problem.

| Class | Forest Cover Type |
|---|---|
| 1 | Spruce/Fir |
| 2 | Lodgepole Pine |
| 3 | Ponderosa Pine |
| 4 | Cottonwood/Willow |
| 5 | Aspen |
| 6 | Douglas-fir |
| 7 | Krummholz |

# 3. Train/Test Split

In [ ]:
# Reuse the shared split loaded above; do not load or split the dataset again.
print("Training:", X_train.shape)
print("Testing:", X_test.shape)


### Train/Test Split

All model experiments use the same seeded, stratified 10,000-row sample, then the same 80/20 stratified train/test split. 

This gives 8,000 training observations and 2,000 held-out test observations in each model experiment.

# 4. Preprocessing and Feature Scaling

In [ ]:
numeric_features = X.columns[:10]

print(numeric_features)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features)
    ],
    remainder="passthrough"
)

### Feature Scaling

The ten numerical features were standardised using StandardScaler. Standardisation is useful for Logistic Regression because the numerical features are measured on very different scales. 

The wilderness area and soil type features are already binary variables, so they were left unchanged.

# Train the Logistic Regression Model

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        random_state=RANDOM_STATE
    ))
])

In [ ]:
model.fit(X_train, y_train)

## Logistic Regression Model

Logistic Regression was selected as the classification model. Although Logistic Regression is commonly introduced as a binary classifier, it can also be used for multiclass classification. 

The preprocessing and Logistic Regression model were combined into a pipeline so that the same preprocessing operations were consistently applied during training and testing.

# 6. Evaluate Model Performance

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

In [ ]:
print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred, labels=np.arange(len(class_names)),
    target_names=class_names, zero_division=0))
print("Held-out predictive performance:", evaluate_predictive_performance(model, X_test, y_test))


## Model performance analysis

Use the classification report and shared predictive metrics above from the current controlled split. Compare macro and balanced metrics alongside accuracy because the classes are imbalanced.


## Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
display_labels=class_names,
    normalize="true",
    cmap="Blues",
    ax=ax
)

plt.title("Normalised Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted Cover Type")
plt.ylabel("Actual Cover Type")
plt.tight_layout()
plt.show()


### Confusion matrix analysis

Inspect which true classes are confused with each predicted class in this controlled run. The matrix and per-class report should be interpreted together.


# 7. Local Explainability with LIME

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

# Use the protocol's background and categorical handling for the illustration too.
categorical_feature_indices = list(range(10, X.shape[1]))
explainer = LimeTabularExplainer(
    training_data=lime_background.to_numpy(),
    feature_names=feature_names,
    class_names=class_names,
    categorical_features=categorical_feature_indices,
    categorical_names={i: ["0", "1"] for i in categorical_feature_indices},
    mode="classification",
    random_state=RANDOM_STATE,
)

def predict_proba_for_lime(rows):
    return model.predict_proba(pd.DataFrame(rows, columns=feature_names))


In [ ]:
# Binary feature handling and LIME background size come from the shared protocol.
print("Categorical positions:", categorical_feature_indices)


In [ ]:
print("Shared LIME background shape:", lime_background.shape)


In [ ]:
# The illustrative explainer was created above using the protocol background and seed.


In [ ]:
# predict_proba_for_lime wraps the entire preprocessing/model pipeline.
# LIME therefore receives raw features in the same units as the other notebooks.


## Explain the first shared held-out case

Use the same protocol row as the other models, whether its prediction is correct or incorrect.


In [ ]:
# Select the same protocol case in every notebook, independently of predictions.
case_index = explanation_cases.index[0]
instance = X_test.loc[case_index]
true_class = int(y_test.loc[case_index])
predicted_class = int(model.predict(X_test.loc[[case_index]])[0])
predicted_class_index = int(np.flatnonzero(model.classes_ == predicted_class)[0])
probabilities = predict_proba_for_lime(instance.to_numpy()[None, :])[0]
print("Shared test row:", case_index)
print("Actual:", class_names[true_class], "| Predicted:", class_names[predicted_class])


In [ ]:
print("Class being explained:", class_names[predicted_class_index])


In [ ]:
# This single explanation illustrates the result; the comparison uses all shared cases.
explanation = explainer.explain_instance(
    data_row=instance.to_numpy(),
    predict_fn=predict_proba_for_lime,
    labels=(predicted_class_index,),
    num_features=NUM_FEATURES,
    num_samples=SAMPLE_BUDGETS[-2],
)


In [ ]:
fig = explanation.as_pyplot_figure(
    label=predicted_class_index
)

plt.title(
    f"LIME Explanation for Predicted Class: "
    f"{class_names[predicted_class_index]}"
)
plt.tight_layout()
plt.show()


## LIME Feature Contributions

In [ ]:
lime_contributions = explanation.as_list(
    label=predicted_class_index
)

for feature, importance in lime_contributions:
    print(f"{feature}: {importance:.4f}")


## Controlled LIME comparison

`controlled_lime_protocol.py` is the source of shared experiment settings. Every model receives the same stratified 10,000-row sample, 8,000 training rows, 2,000 held-out rows, 54 input predictors, and labels 0–6. The protocol chooses three held-out cases per true class without filtering by predictions, then uses the same 2,000-row training background, budgets, five repeat seeds (42–46), ten explanation terms, and stability definitions.

The 18/36/54-feature subsets change which predictors LIME perturbs. Models are fitted once on all 54 predictors; excluded predictors remain fixed at each case's value. Runtime includes LIME's prediction calls and surrogate fitting but excludes explainer setup and prediction warm-up. Fidelity is the local surrogate's weighted R², not predictive accuracy. Stability uses signed-weight Spearman on selected-feature unions and top-five Jaccard.

Model architecture, its hyperparameters and required preprocessing remain model-specific. Compare timing on the same hardware and backend where possible; CUDA and CPU timings have different resource conditions. Results below must be regenerated after changing the protocol.


In [ ]:
# The shared function owns sampling, seeds, feature subsets and stability calculations.
model_name = 'LogisticRegression'
controlled_raw, controlled_summary = benchmark_lime(
    model, X_train, X_test, y_test, predict_proba_for_lime, model_name
)
predictive_metrics = evaluate_predictive_performance(model, X_test, y_test)
print("Held-out predictive performance:", predictive_metrics)
display(controlled_summary.round(4))

# Write beside the protocol, regardless of whether Jupyter started at the repo root.
import controlled_lime_protocol as protocol
output_directory = Path(protocol.__file__).resolve().parent
controlled_raw.to_csv(output_directory / f"{model_name.lower()}_controlled_lime_raw.csv", index=False)
controlled_summary.to_csv(output_directory / f"{model_name.lower()}_controlled_lime_summary.csv", index=False)
pd.DataFrame([{"model": model_name, **predictive_metrics}]).to_csv(
    output_directory / f"{model_name.lower()}_controlled_predictive_metrics.csv", index=False
)


In [ ]:
# Plot the shared summary separately for each perturbation dimension.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for feature_count, group in controlled_summary.groupby("feature_count"):
    group = group.sort_values("num_samples")
    label = f"{feature_count} perturbed features"
    axes[0, 0].errorbar(group["num_samples"], group["runtime_mean_seconds"],
        yerr=group["runtime_std_seconds"], marker="o", capsize=3, label=label)
    axes[0, 1].errorbar(group["num_samples"], group["fidelity_mean"],
        yerr=group["fidelity_std"], marker="o", capsize=3, label=label)
    axes[1, 0].plot(group["num_samples"], group["spearman_stability_mean"], marker="o", label=label)
    axes[1, 1].plot(group["num_samples"], group["top5_jaccard_mean"], marker="o", label=label)

for ax, title, ylabel in zip(axes.flat,
    ["Explanation runtime", "Local fidelity", "Signed-weight rank stability", "Top-five feature stability"],
    ["Seconds (mean ± SD)", "Weighted surrogate R² (mean ± SD)", "Mean pairwise Spearman", "Mean pairwise Jaccard"]):
    ax.set(title=title, xlabel="Perturbation samples", ylabel=ylabel, xscale="log")
    ax.set_xticks(SAMPLE_BUDGETS, labels=[f"{budget:,}" for budget in SAMPLE_BUDGETS])
    ax.tick_params(axis="x", rotation=30)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0, 0].set_ylim(bottom=0)
axes[1, 0].set_ylim(-1.05, 1.05)
axes[1, 1].set_ylim(-0.05, 1.05)
fig.suptitle(f"{model_name}: controlled LIME comparison")
plt.tight_layout()
plt.show()


### Interpreting the controlled experiment

Compare matching perturbation dimensions and sample budgets across models. Runtime is per explanation; fidelity is weighted local R². Spearman measures signed-weight rank consistency, and top-five Jaccard measures selected-feature overlap. Both stability measures compare repeated explanations of the same case before averaging across cases.


## What LIME Is Showing
LIME (Local Interpretable Model-agnostic Explanations) is used here to explain one individual prediction made by the trained Logistic Regression pipeline. 

LIME creates perturbed samples around the selected observation, asks the model for predictions on those samples, and fits a simpler interpretable model in that local neighbourhood. The resulting weights show which features locally increased or decreased support for the class being explained. 

This is a local explanation. It describes the model's behaviour around this particular observation and should not be interpreted as a global statement about every prediction in the dataset.

## LIME findings

The illustration explains the first protocol case's predicted class, independently of prediction correctness. Positive weights support that class locally and negative weights oppose it. Use the controlled summary for comparisons across models; this single-row plot is illustrative only.


# 8. Global Explainability with Logistic Regression Coefficients

In [ ]:
classifier = model.named_steps["classifier"]

transformed_feature_names = (
    model.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Remove ColumnTransformer prefixes to make the table easier to read.
clean_feature_names = [
    name.replace("numeric__", "").replace("remainder__", "")
    for name in transformed_feature_names
]

coefficients = pd.DataFrame(
    classifier.coef_,
    columns=clean_feature_names,
    index=class_names
)

coefficients


In [ ]:
spruce_coefficients = (
    coefficients.loc["Spruce/Fir"]
    .sort_values(key=abs, ascending=False)
    .head(10)
)

spruce_coefficients


### Global Explanation Findings
Logistic Regression is relatively interpretable because its learned coefficients can be inspected directly. These coefficients provide a global view of the model rather than explaining a single observation. 

The table above lists the ten largest coefficients by absolute magnitude for the Spruce/Fir class. A positive coefficient raises the model's linear score for Spruce/Fir as that feature increases or is present, while a negative coefficient lowers that score, with the other features held fixed. 

The coefficient magnitudes should be interpreted carefully because they describe the fitted model rather than proving a real-world causal relationship.

# 9. Local vs Global Explainability 

The two explanation approaches answer different questions:
- LIME is local: it explains why the model made one particular prediction.
- Logistic Regression coefficients are global: they describe relationships learned across the fitted model as a whole. 

A feature can be important globally without being one of the strongest influences for every individual prediction. Using both approaches therefore gives a more complete understanding of the classifier than performance metrics alone.

# 10. Conclusion

Logistic Regression is fitted on the shared 8,000 training observations and evaluated on the same 2,000 held-out observations as the other model notebooks. Numerical scaling is fitted only on training data inside the pipeline. Model-specific preprocessing is part of the fitted model being compared.

Use the freshly generated classification metrics and controlled LIME summary to report performance. Results from the earlier full-dataset experiment do not describe this controlled configuration.


# 11. Summary report

The shared protocol fixes the sampled rows, split, label mapping, explanation cases, background, perturbation budgets, feature subsets and repeat seeds. This notebook changes the fitted model to a Logistic Regression pipeline. Predictive metrics describe classification performance; LIME runtime, weighted local R² and stability describe the explanations. The exported CSV files contain the current results after running the notebook.


## Local and global explanations

The LIME weights explain individual predictions. Logistic Regression coefficients describe the fitted classifier across observations. Neither explanation proves causation; use both alongside the current predictive metrics.
